In [3]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

# This is required before importingJAX in order to run multiple processes.
from multiprocessing import set_start_method
set_start_method('forkserver', force=True)

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from tqdm import tqdm
from pathlib import Path

# This increases run time but makes troubleshooting much easier
jax.config.update("jax_debug_nans", True) 

print(jax.devices())

[CpuDevice(id=0)]


In [5]:
from config import Config
cfg = Config.from_file("/nas/cee-water/cjgleason/ted/swot-ml/runs/reservoir_sf_grnn/era5.yml")
cfg.quiet = False # Show more output for testing

# Printing will show all options (including the default ones not set in your yml file)
print(cfg)

{
    "data_root": "/nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/Columbia",
    "zarr_dir": "/nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/Columbia/zarr_store",
    "attributes_file": "/nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/Columbia/attributes/static_4km2.parquet",
    "train_basin_file": "/nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/Columbia/basin_lists/Columbia/4km2/train.csv",
    "test_basin_file": "/nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/Columbia/basin_lists/Columbia/4km2/test.csv",
    "graph_network_file": "/nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/Columbia/metadata/graphs_4km2.json",
    "features": {
        "dynamic": {
            "era5": [
                "ssrd_mean",
                "strd_mean",
                "ro_mean",
                "sro_mean",
                "e_mean",
                "tp_mean",
                "sp_mean",
                "d2m_mean",
                "t2m_mean",


In [6]:
import data 

# This sets up a prescaled zarr cache based on the features, encodings, and normalization scheme
mgr = data.DynamicCacheManager(cfg) 
cache_dir = mgr.create_cache('train')

Caches will be stored at: /nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/Columbia/zarr_store/_cache4/e9c272275c122381
Indices for train already exist. Skipping.


In [7]:
# the `dataset` object interacts with the cache when the dataloader requests a sample
dataset = data.CachedBasinGraphDataset(cfg, cache_dir, 'train')

# the `dataloader` is the iterator which will return batches of data for training
cfg.num_workers = 0 # faster for a single batch
dataloader = data.CachedBasinGraphDataLoader(cfg, dataset)

Calculating training statistics for encoding and normalization...
Loading basin graphs...Done!
Loading optimized indices for train...Done!
Dataloader using 0 parallel CPU worker(s).


In [8]:
import train

# Default is make a run directory next to the config file. This 
log_dir = Path('test_run').resolve() 

# You can also just turn off logging if you are testing stuff in a notebook/terminal and don't need to save files.
# cfg.log = False

trainer = train.Trainer(cfg, dataloader, log_dir=log_dir)

Model contains 419,256 parameters
Logging at /nas/cee-water/cjgleason/ted/swot-ml/notebooks/demo/test_run


In [9]:
trainer.model

SparseFusionGRNN(
  head={
    'discharge':
    GMM(
      fc1=Linear(
        weight=f32[128,64],
        bias=f32[128],
        in_features=64,
        out_features=128,
        use_bias=True
      ),
      fc2=Linear(
        weight=f32[300,128],
        bias=f32[300],
        in_features=128,
        out_features=300,
        use_bias=True
      ),
      _eps=0.001
    ),
    'dv_SWOTAE':
    GMM(
      fc1=Linear(
        weight=f32[128,64],
        bias=f32[128],
        in_features=64,
        out_features=128,
        use_bias=True
      ),
      fc2=Linear(
        weight=f32[300,128],
        bias=f32[300],
        in_features=128,
        out_features=300,
        use_bias=True
      ),
      _eps=0.001
    )
  },
  target=['discharge', 'dv_SWOTAE'],
  backbone_leaves=['dense_embedders', 'fusion_norm', 'spat_temp_lstm', 'head'],
  hidden_size=64,
  seq_length=90,
  dense_sources=['era5'],
  static_size=132,
  seq2seq=False,
  supervised_attn=False,
  use_obs_memory=True,
  r

In [ ]:
# a single step. Normally you would just call it without args and it will go until max_training_steps or max_training_hours 
trainer.start_training(stop_at=1) 

# It might take a long time or even crash on CPU without enough RAM. 
# You can start an interactive session with a GPU if you really want to test here, although the next notebook shows you how to submit sbatch jobs to the gpu cluster.

2026-04-16 13:34:12,316 - INFO - ~~~ Starting training (max_steps=1, max_hours=336) ~~~


INFO:training:~~~ Starting training (max_steps=1, max_hours=336) ~~~
Training Steps: 36it [19:37, 39.05s/it]                      